# Notebook Huấn luyện & Đánh giá Mô hình Phân loại (Classification Notebook)

Notebook này thực hiện huấn luyện, tinh chỉnh và đánh giá so sánh hiệu năng các thuật toán học máy phân loại trễ chuyến bay trên dữ liệu chuẩn hóa của dự án Aeolus:
1. **Bài toán 1 (Cốt lõi - Downstream Gate Assignment)**: **Phân loại Đến trễ (`IS_ARR_DELAY = 1[ARR_DELAY >= 15]`)**
   * Mốc dự báo an toàn: $T-2\text{h}$ trước khi khởi hành từ sân bay gốc, loại trừ hoàn toàn rò rỉ dữ liệu.
   * Cung cấp xác suất trễ phục vụ bộ giải tối ưu phân bổ cổng tại sân bay Atlanta (ATL).
2. **Bài toán 2 (Phụ trợ - Outbound Study)**: **Phân loại Khởi hành trễ (`IS_DEP_DELAY = 1[DEP_DELAY >= 15]`)**

### 🤖 Các thuật toán được đánh giá ngang hàng:
* **XGBoost Classifier**: Mô hình Gradient Boosting hiệu năng cao (tự động kích hoạt GPU CUDA hoặc fallback CPU đa luồng).
* **Logistic Regression**: Mô hình tuyến tính chuẩn có trọng số cân bằng lớp (`class_weight='balanced'`).
* **Random Forest Classifier**: Mô hình kết hợp biểu quyết cây quyết định đa luồng.

### Cell 1: Import các thư viện cần thiết & Thiết lập môi trường

In [1]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    auc,
    accuracy_score,
    f1_score,
    confusion_matrix
)

# Cấu hình cảnh báo và hiển thị
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)

# Hàm xác định thư mục gốc của repository
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "data" / "split").exists() or (candidate / "src" / "data").exists():
            return candidate
    return Path.cwd()

repo_root = resolve_repo_root()
print(f"Repository Root: {repo_root}")
print("Tất cả các thư viện phân loại đã được import thành công!")

Repository Root: d:\Documents\BaiTap\KhoaLuanCuNhan\aeolus-gate-optimization-1\src
Tất cả các thư viện phân loại đã được import thành công!


### Cell 2: Hàm Nạp Dữ liệu Phân vùng Splits (Data Loader)
Hàm tự động nạp các tệp Parquet đã được chia sẵn theo mốc thời gian từ `preprocessing_notebook.ipynb`:
* **Train Set**: 2016 – 2022
* **Validation Set**: 2023
* **Test Set**: 2024 (Sealed Holdout)

In [3]:
def load_split_dataset(task_name):
    """
    Nạp dữ liệu từ thư mục split được tạo bởi preprocessing_notebook.
    Đọc X_*.parquet và y_*.parquet một cách ĐỘC LẬP (không đọc thư mục chung)
    để tránh lỗi schema mismatch giữa file X và file y.
    """
    candidate_base_dirs = [
        repo_root / "data" / "split" / task_name,
        repo_root / "src" / "data" / "split" / task_name
    ]
    
    task_dir = None
    for d in candidate_base_dirs:
        if d.exists():
            task_dir = d
            break
            
    if task_dir is None:
        raise FileNotFoundError(
            f"Không tìm thấy thư mục split '{task_name}'! "
            f"Vui lòng chạy 'preprocessing_notebook.ipynb' trước để tạo dữ liệu."
        )
        
    print(f"-> Đang nạp dữ liệu từ: {task_dir}")
    
    def load_fold(fold_name):
        fold_dir = task_dir / fold_name
        # Đọc riêng biệt X_*.parquet và y_*.parquet để tránh schema mismatch
        x_files = sorted(fold_dir.glob("X_*.parquet"))
        y_files = sorted(fold_dir.glob("y_*.parquet"))
        
        if not x_files or not y_files:
            raise FileNotFoundError(f"Không tìm thấy X_*.parquet hoặc y_*.parquet trong {fold_dir}")
        
        X = pd.concat([pd.read_parquet(f) for f in x_files], ignore_index=True)
        y = pd.concat([pd.read_parquet(f) for f in y_files], ignore_index=True)["target"]
        
        # Loại bỏ target column nếu vô tình có trong X
        drop_targets = [c for c in ["target", "IS_ARR_DELAY", "IS_DEP_DELAY", "ARR_DELAY", "DEP_DELAY"] if c in X.columns]
        if drop_targets:
            X = X.drop(columns=drop_targets)
        
        return X, y
    
    X_train, y_train = load_fold("train")
    X_valid, y_valid = load_fold("valid")
    X_test,  y_test  = load_fold("test")
        
    print(f"   * Kích thước Train (2016-2022) : X={X_train.shape}, y={y_train.shape} (Tỷ lệ trễ: {y_train.mean()*100:.2f}%)")
    print(f"   * Kích thước Valid (2023)      : X={X_valid.shape}, y={y_valid.shape} (Tỷ lệ trễ: {y_valid.mean()*100:.2f}%)")
    print(f"   * Kích thước Test  (2024)      : X={X_test.shape}, y={y_test.shape} (Tỷ lệ trễ: {y_test.mean()*100:.2f}%)")
    
    return X_train, y_train, X_valid, y_valid, X_test, y_test

print("Hàm load_split_dataset đã sẵn sàng!")

Hàm load_split_dataset đã sẵn sàng!


### Cell 3: Hàm Huấn luyện & Đánh giá Mô hình Phân loại (train_and_evaluate_classifier)
* **Chuẩn hóa StandardScaler**: Chỉ `fit` trên tập `X_train` để loại trừ hoàn toàn rò rỉ phân phối (Data Leakage).
* **Xử lý bất đối xứng tài nguyên phần cứng**: Tự động nhận diện GPU NVIDIA qua `device='cuda'`, fallback an toàn về CPU đa luồng (`n_jobs=-1`) nếu không có card GPU.
* **Đo lường đa chỉ số**: `ROC-AUC`, `PR-AUC`, `Accuracy`, `F1-Score`, và bảng chi tiết `classification_report`.

In [4]:
all_classification_metrics = []

def train_and_evaluate_classifier(X_train, y_train, X_valid, y_valid, X_test, y_test, task_label, model_types=['xgb', 'lr', 'rf']):
    print(f"\n{'='*70}")
    print(f"BẮT ĐẦU HUẤN LUYỆN BÀI TOÁN PHÂN LOẠI: {task_label.upper()}")
    print(f"CÁC THUẬT TOÁN: {model_types}")
    print(f"{'='*70}")
    
    # Chuẩn hóa các biến số liên tục (chỉ fit trên train để tránh rò rỉ dữ liệu)
    scaler = StandardScaler()
    X_train_scaled = X_train.copy()
    X_valid_scaled = X_valid.copy()
    X_test_scaled  = X_test.copy()
    
    numeric_cols = [c for c in X_train.columns if X_train[c].dtype in ['float32', 'float64']]
    if numeric_cols:
        X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
        X_valid_scaled[numeric_cols] = scaler.transform(X_valid[numeric_cols])
        X_test_scaled[numeric_cols]  = scaler.transform(X_test[numeric_cols])
        
    trained_models = {}
    
    for m_type in model_types:
        print(f"\n{'-'*50}")
        print(f"Đang cấu hình và huấn luyện mô hình: {m_type.upper()}...")
        
        if m_type == 'xgb':
            import xgboost as xgb
            # Thử nghiệm GPU CUDA, nếu không có thì fallback sang CPU
            use_device = 'cuda'
            n_neg = int((y_train == 0).sum())
            n_pos = int((y_train == 1).sum())
            spw = round(n_neg / n_pos, 2)
            print(f"-> Class imbalance ratio (scale_pos_weight): {spw:.2f}")
            try:
                test_clf = xgb.XGBClassifier(n_estimators=1, device='cuda')
                test_clf.fit(X_train_scaled.iloc[:5], y_train.iloc[:5])
                print("-> Kích hoạt thành công tăng tốc GPU (CUDA) cho XGBoost!")
            except Exception:
                print("-> GPU không khả dụng, chuyển sang sử dụng CPU đa luồng...")
                use_device = 'cpu'
                
            clf = xgb.XGBClassifier(
                n_estimators=150,
                max_depth=8,
                learning_rate=0.08,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                tree_method='hist',
                device=use_device,
                scale_pos_weight=spw,
                n_jobs=-1 if use_device == 'cpu' else None
            )
        elif m_type == 'lr':
            # saga solver hỗ trợ n_jobs=-1 (song song hóa)
            clf = LogisticRegression(
                max_iter=500,
                class_weight='balanced',
                solver='saga',
                random_state=42,
                n_jobs=-1
            )
        elif m_type == 'rf':
            clf = RandomForestClassifier(
                n_estimators=100,
                max_depth=12,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            )
        else:
            print(f"Bỏ qua: Mô hình '{m_type}' không hợp lệ.")
            continue
            
        # Huấn luyện mô hình
        clf.fit(X_train_scaled, y_train)
        
        # Đánh giá trên tập Validation (2023)
        y_val_prob = clf.predict_proba(X_valid_scaled)[:, 1]
        val_auc = roc_auc_score(y_valid, y_val_prob)
        print(f"Validation ROC-AUC (2023): {val_auc:.4f}")
        
        # Đánh giá trên tập Test (2024)
        y_test_pred = clf.predict(X_test_scaled)
        y_test_prob = clf.predict_proba(X_test_scaled)[:, 1]
        
        test_auc = roc_auc_score(y_test, y_test_prob)
        test_acc = accuracy_score(y_test, y_test_pred)
        test_f1  = f1_score(y_test, y_test_pred, zero_division=0)
        
        # Tính PR-AUC (Precision-Recall AUC - chỉ số then chốt cho dữ liệu lệch nhãn)
        p_curve, r_curve, _ = precision_recall_curve(y_test, y_test_prob)
        test_prauc = auc(r_curve, p_curve)
        
        print(f"Test ROC-AUC Score (2024): {test_auc:.4f} | PR-AUC: {test_prauc:.4f}")
        print(f"Test Accuracy: {test_acc*100:.2f}% | Test F1-Score: {test_f1:.4f}")
        print(f"\n--- Báo cáo chi tiết phân loại {m_type.upper()} trên tập Test (2024) ---")
        print(classification_report(y_test, y_test_pred, digits=4))
        
        all_classification_metrics.append({
            "Task": task_label,
            "Model": m_type.upper(),
            "Val ROC-AUC": round(val_auc, 4),
            "Test ROC-AUC": round(test_auc, 4),
            "Test PR-AUC": round(test_prauc, 4),
            "Test Accuracy": f"{test_acc*100:.2f}%",
            "Test F1-Score": round(test_f1, 4)
        })
        
        trained_models[m_type] = clf
        
    return trained_models

print("Hàm train_and_evaluate_classifier đã sẵn sàng!")

Hàm train_and_evaluate_classifier đã sẵn sàng!


### Cell 4: Thực nghiệm Bài toán 1 (Cốt lõi) - Dự đoán Đến trễ (IS_ARR_DELAY >= 15)
Dự báo xác suất chuyến bay đến sân bay Atlanta bị trễ từ 15 phút trở lên tại thời điểm trước khi cất cánh ($T-2\text{h}$).

In [6]:
# Nạp dữ liệu bài toán Đến trễ
X_tr_arr, y_tr_arr, X_va_arr, y_va_arr, X_te_arr, y_te_arr = load_split_dataset("arrival_classification")

# Huấn luyện và so sánh mô hình
models_arrival = train_and_evaluate_classifier(
    X_tr_arr, y_tr_arr,
    X_va_arr, y_va_arr,
    X_te_arr, y_te_arr,
    task_label="IS_ARR_DELAY >= 15",
    model_types=['xgb', 'lr', 'rf']
)

-> Đang nạp dữ liệu từ: d:\Documents\BaiTap\KhoaLuanCuNhan\aeolus-gate-optimization-1\src\data\split\arrival_classification
   * Kích thước Train (2016-2022) : X=(41743574, 22), y=(41743574,) (Tỷ lệ trễ: 17.83%)
   * Kích thước Valid (2023)      : X=(6645342, 22), y=(6645342,) (Tỷ lệ trễ: 20.54%)
   * Kích thước Test  (2024)      : X=(6284739, 22), y=(6284739,) (Tỷ lệ trễ: 20.81%)

BẮT ĐẦU HUẤN LUYỆN BÀI TOÁN PHÂN LOẠI: IS_ARR_DELAY >= 15
CÁC THUẬT TOÁN: ['xgb', 'lr', 'rf']

--------------------------------------------------
Đang cấu hình và huấn luyện mô hình: XGB...
-> Class imbalance ratio (scale_pos_weight): 4.61
-> Kích hoạt thành công tăng tốc GPU (CUDA) cho XGBoost!
Validation ROC-AUC (2023): 0.6603
Test ROC-AUC Score (2024): 0.6603 | PR-AUC: 0.3306
Test Accuracy: 57.67% | Test F1-Score: 0.3997

--- Báo cáo chi tiết phân loại XGB trên tập Test (2024) ---
              precision    recall  f1-score   support

           0     0.8665    0.5504    0.6732   4977197
           1     

MemoryError: Unable to allocate 318. MiB for an array with shape (333948592,) and data type uint8

### Cell 5: Thực nghiệm Bài toán 2 (Phụ trợ) - Dự đoán Khởi hành trễ (IS_DEP_DELAY >= 15)
Dự báo trễ cất cánh phục vụ nghiên cứu độc lập đối chứng.

In [ ]:
# Nạp dữ liệu bài toán Khởi hành trễ
X_tr_dep, y_tr_dep, X_va_dep, y_va_dep, X_te_dep, y_te_dep = load_split_dataset("departure_classification")

# Huấn luyện và so sánh mô hình
models_departure = train_and_evaluate_classifier(
    X_tr_dep, y_tr_dep,
    X_va_dep, y_va_dep,
    X_te_dep, y_te_dep,
    task_label="IS_DEP_DELAY >= 15",
    model_types=['xgb', 'lr', 'rf']
)

### Cell 6: Bảng So sánh Tổng hợp Hiệu năng các Mô hình Phân loại
Tổng kết và xếp hạng các thuật toán theo ROC-AUC, PR-AUC, Accuracy và F1-Score trên cả 2 bài toán dự báo.

In [ ]:
df_summary_cls = pd.DataFrame(all_classification_metrics)
print("=" * 85)
print("TỔNG HỢP SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH PHÂN LOẠI (CLASSIFICATION BENCHMARK)")
print("=" * 85)
print(df_summary_cls.to_string(index=False))

### Cell 7: Trực quan hóa Tầm quan trọng của Đặc trưng (Feature Importance Analysis)
Trích xuất và trực quan hóa top 15 đặc trưng có đóng góp cao nhất vào quyết định phân loại trễ chuyến của mô hình XGBoost.

In [ ]:
if 'xgb' in models_arrival:
    xgb_model = models_arrival['xgb']
    importances = xgb_model.feature_importances_
    features = X_tr_arr.columns
    
    indices = np.argsort(importances)[::-1][:15]
    
    plt.figure(figsize=(10, 6))
    plt.title("Top 15 Đặc trưng quan trọng nhất - XGBoost (Bài toán Đến trễ)")
    plt.barh(range(15), importances[indices][::-1], color='steelblue', align="center")
    plt.yticks(range(15), [features[i] for i in indices][::-1])
    plt.xlabel("Mức độ quan trọng (Feature Importance Gain)")
    plt.tight_layout()
    plt.show()